In [5]:
import pandas as pd
import numpy as np

# 1. Load the datasets
sales = pd.read_csv('../data/sales_train_evaluation.csv')
calendar = pd.read_csv('../data/calendar.csv')
prices = pd.read_csv('../data/sell_prices.csv')

In [10]:
print(f"Full sales shape: {sales.shape}")

Full sales shape: (30490, 1947)


In [11]:
sales_ca1 = sales[sales['store_id'] == 'CA_1'].copy()
print(f"CA_1 sales shape: {sales_ca1.shape}") # Should be ~3,049 SKUs


CA_1 sales shape: (3049, 1947)


In [12]:
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
df = pd.melt(
    sales_ca1, 
    id_vars=id_cols, 
    var_name='d', 
    value_name='sales'
)


In [13]:
df = df.merge(calendar[['d', 'date', 'wm_yr_wk', 'event_name_1', 'event_type_1', 'snap_CA']], on='d', how='left')
df = df.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')


In [15]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['id', 'date']).reset_index(drop=True)

print(df.head())
print(df.info())

                            id      item_id  dept_id cat_id store_id state_id  \
0  FOODS_1_001_CA_1_evaluation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
1  FOODS_1_001_CA_1_evaluation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
2  FOODS_1_001_CA_1_evaluation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
3  FOODS_1_001_CA_1_evaluation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   
4  FOODS_1_001_CA_1_evaluation  FOODS_1_001  FOODS_1  FOODS     CA_1       CA   

     d  sales       date  wm_yr_wk event_name_1 event_type_1  snap_CA  \
0  d_1      3 2011-01-29     11101          NaN          NaN        0   
1  d_2      0 2011-01-30     11101          NaN          NaN        0   
2  d_3      0 2011-01-31     11101          NaN          NaN        0   
3  d_4      1 2011-02-01     11101          NaN          NaN        1   
4  d_5      4 2011-02-02     11101          NaN          NaN        1   

   sell_price  
0         2.0  
1         2.0  
2         2.0  
3         

In [16]:
# Check how many rows are pre-launch (missing sell_price)
missing_prices = df['sell_price'].isna().sum()
print(f"Pre-launch rows to drop: {missing_prices:,} ({missing_prices/len(df):.1%} of data)")

# Drop those rows so we only model active SKUs
df = df.dropna(subset=['sell_price']).reset_index(drop=True)
print(f"Cleaned dataset shape: {df.shape}")


Pre-launch rows to drop: 1,129,842 (19.1% of data)
Cleaned dataset shape: (4788267, 14)


In [17]:
# Total units sold and % share by category
cat_summary = df.groupby('cat_id')['sales'].agg(
    total_units='sum',
    avg_daily_sales='mean'
)
cat_summary['pct_share'] = (cat_summary['total_units'] / cat_summary['total_units'].sum()) * 100
print(cat_summary)


           total_units  avg_daily_sales  pct_share
cat_id                                            
FOODS          5471661         2.420831  69.860671
HOBBIES         892083         1.003741  11.389872
HOUSEHOLD      1468504         0.895829  18.749457


In [18]:
# Percentage of SKU-days with zero sales
zero_days = (df['sales'] == 0).mean() * 100
print(f"Percentage of SKU-days with 0 sales: {zero_days:.2f}%")


Percentage of SKU-days with 0 sales: 55.21%


In [23]:
# Average sales by day of the week
!pip install pyarrow

   ---------------------------------------- 0.0/29.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/29.0 MB ? eta -:--:--
   ----- ---------------------------------- 3.7/29.0 MB 15.3 MB/s eta 0:00:02
   ------------- -------------------------- 9.4/29.0 MB 21.2 MB/s eta 0:00:01
   ---------------------- ----------------- 16.0/29.0 MB 23.9 MB/s eta 0:00:01
   -------------------------- ------------- 19.1/29.0 MB 22.2 MB/s eta 0:00:01
   --------------------------------- ------ 24.4/29.0 MB 22.5 MB/s eta 0:00:01
   ---------------------------------------  28.8/29.0 MB 22.5 MB/s eta 0:00:01
   ---------------------------------------- 29.0/29.0 MB 21.6 MB/s  0:00:01



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
# Average sales by day of the week
df['day_name'] = df['date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
print(df.groupby('day_name')['sales'].mean().reindex(day_order))

# Save cleaned dataframe for Step 3 (Feature Engineering)
df.to_pickle('../data/ca1_cleaned.pkl')
print("Saved ca1_cleaned.pkl successfully!")

day_name
Monday       1.567289
Tuesday      1.386968
Wednesday    1.350030
Thursday     1.365567
Friday       1.634924
Saturday     2.057239
Sunday       2.084118
Name: sales, dtype: float64
Saved ca1_cleaned.pkl successfully!
